In [12]:
from typing import TypedDict,Annotated
from operator import add

from langchain_classic.tools import sleep
from langgraph import graph
from langgraph.graph import StateGraph,START,END
from langgraph.types import Overwrite

#1. 定义节点状态
class OverAllState(TypedDict):
    #归约的方式是追加合并
    loges:Annotated[list[str],add]
    #如果出现并行结果往下游节点传递必须要有reducer函数
    cur_id: Annotated[str,add]

#2. 定义节点
def node_1(state:OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"1k:{k}v:{v}")
    return {
        "loges":["node_1运行完毕"],
    }

def node_2(state:OverAllState) -> OverAllState:
    for k,v in state.items():
        print(f"2k:{k}v:{v}")
    return {
        "loges":["node_2运行完毕"],
        "cur_id":"node2"
    }


def node_3(state:OverAllState) -> OverAllState:

    for k,v in state.items():
        print(f"3k:{k}v:{v}")
    return {
        "loges":["node_3运行完毕"],
        "cur_id":"node3"
    }

def node_4(state:OverAllState) -> OverAllState:

    for k,v in state.items():
        print(f"4k:{k}v:{v}")
    return {
        "loges":["node_4运行完毕"],
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1",node_1)
builder.add_node("node_2",node_2)
builder.add_node("node_3",node_3)
builder.add_node("node_4",node_4)
builder.add_edge(START,"node_1")
builder.add_edge("node_1","node_2")
builder.add_edge("node_1","node_3")
builder.add_edge("node_2","node_4")
builder.add_edge("node_3","node_4")
builder.add_edge("node_4",END)

graph = builder.compile()

result = graph.invoke({"logs":["START"]})
print('='* 30,'->result<-','=' *30)
print(result)


1k:logesv:[]
1k:cur_idv:
2k:logesv:['node_1运行完毕']
2k:cur_idv:
3k:logesv:['node_1运行完毕']
3k:cur_idv:
4k:logesv:['node_1运行完毕', 'node_2运行完毕', 'node_3运行完毕']
4k:cur_idv:node2node3
============================== ->result<- ==============================
{'loges': ['node_1运行完毕', 'node_2运行完毕', 'node_3运行完毕', 'node_4运行完毕'], 'cur_id': 'node2node3'}
